In [33]:
class Card:
    """
    Represents a single UNO card.
    color: 'Red', 'Blue', 'Green', 'Yellow'
    value: 0-9 or 'Skip'
    """
    def __init__(self, color, value):
        self.color = color
        self.value = value

    def __repr__(self):
        return f"{self.color} {self.value}"

    def is_skip(self):
        return self.value == 'Skip'

    def matches(self, top_card):
        return self.color == top_card.color or self.value == top_card.value

c1 = Card('Red', 5)
c2 = Card('Blue', 5)
c3 = Card('Green', 3)
print(c1, 'matches', c2, '->', c1.matches(c2))
print(c1, 'matches', c3, '->', c1.matches(c3))

Red 5 matches Blue 5 -> True
Red 5 matches Green 3 -> False


In [35]:
import random
def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    deck = []
    for color in colors:
        for number in range(10):
            deck.append(Card(color, number))
    for color in colors:
        deck.append(Card(color, 'Skip'))
    random.shuffle(deck)
    return deck

def get_valid_moves(hand, top_card):
    return [card for card in hand if card.matches(top_card)]

def create_initial_state():
    deck = generate_deck()
    hands = [[], [], []]
    for _ in range(5):
        for p in range(3):
            hands[p].append(deck.pop())
    top_card = deck.pop()
    while top_card.is_skip():
        deck.insert(0, top_card)
        top_card = deck.pop()
    return {'hands': hands, 'top_card': top_card, 'deck': deck, 'current': 0, 'skip_next': False}

In [37]:
def apply_move(state, player_idx, card):
    new_state = {
        'hands': [list(h) for h in state['hands']],
        'top_card': state['top_card'],
        'deck': list(state['deck']),
        'current': state['current'],
        'skip_next': False,
        'winner': state.get('winner', None)
    }
    if card is None:
        if new_state['deck']:
            new_state['hands'][player_idx].append(new_state['deck'].pop())
    else:
        new_state['hands'][player_idx].remove(card)
        new_state['top_card'] = card
        if card.is_skip():
            new_state['skip_next'] = True
        if len(new_state['hands'][player_idx]) == 0:
            new_state['winner'] = player_idx

    next_player = (player_idx + 1) % 3
    if new_state['skip_next']:
        next_player = (player_idx + 2) % 3
        new_state['skip_next'] = False
    new_state['current'] = next_player
    return new_state

random.seed(42)
state = create_initial_state()
print(f"Before - P1 hand: {state['hands'][0]}")
valid = get_valid_moves(state['hands'][0], state['top_card'])
print(f"Valid moves: {valid}")
if valid:
    state2 = apply_move(state, 0, valid[0])
    print(f"After playing {valid[0]} - P1 hand: {state2['hands'][0]}")
    print(f"New top card: {state2['top_card']}")

Before - P1 hand: [Red Skip, Blue 7, Red 8, Red 5, Blue Skip]
Valid moves: [Blue 7, Blue Skip]
After playing Blue 7 - P1 hand: [Red Skip, Red 8, Red 5, Blue Skip]
New top card: Blue 7
